[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Field Types and Defaults


## What you will be able to do

Write a model whose fields are dates, times, decimals, enumerated values, booleans and UUIDs, and
read the column each one became. Tell a default that lives in Python from one that lives in the
table, and know which rows get which. Read a value back and see what survived the round trip through
SQLite and what did not, the timezone in particular. Recognize the failures that belong to types: a
field Python has no column for, a column whose default only the model knows, a date that arrived as
text, and a row that was written happily and cannot be read back at all.


## The idea

### The problem

So far every field has been a `str`, an `int` or a `bool`, and the columns have looked after
themselves. Real rows are not like that. A mission starts on a date, was filed at a moment, pays a
reward in money, has a status from a short list of allowed values, and is identified by something
that can be made before the database sees it. Every one of those is a Python type that a database has
to be told how to store.

The mapping is mostly quiet and occasionally wrong. `date` and `datetime` have columns of their own.
Money in a `float` loses cents, which is why `Decimal` exists and why it needs to be told how many
digits. A short list of values can be a string column or a real enumerated type. A `dict` has no
column at all, and the class refuses to be defined.

Then there are defaults, which look like one idea and are two. `Field(default=...)` and
`default_factory` are Python's: the model fills the value in when nothing was passed, and the table
knows nothing about it. A column default is the table's: it fills the value in for anything that
writes a row, including a migration, another program, or a line of SQL somebody typed. Which one you
want depends on who writes rows, and getting it wrong is silent until the day something other than
your models writes one.

### What a field says about its column

> The **annotation** picks the column type: `str` is `VARCHAR`, `int` is `INTEGER`, `date` is `DATE`,
> `datetime` is `DATETIME`, `bool` is `BOOLEAN`, a `str` **`Enum`** is a `VARCHAR` long enough for
> its longest value, `uuid.UUID` is a `CHAR(32)`, and `Decimal` is a `NUMERIC`. `| None` decides
> whether the column accepts a null. **`Field`** carries the rest: `max_length` is the `VARCHAR`'s
> length, `max_digits` and `decimal_places` are the `NUMERIC`'s, `unique=True` is a constraint in the
> table, `index=True` is a separate `CREATE INDEX`, and **`default`** and **`default_factory`** are
> Python's defaults, while a column's own default is written with
> `sa_column_kwargs={"server_default": ...}`.

### Why it works that way

- **The annotation is the only place the type is written.** There is no second declaration to keep in
  step, which also means a change to an annotation is a change to the schema, and `create_all` will
  not make it. The **Migrations** notebook is where that goes.
- **A Python default never reaches the table.** The model fills it in when it builds an object, so a
  row written any other way has nothing, and a `NOT NULL` column then refuses it.
- **A column default reaches everything.** It is in the `CREATE TABLE`, so every writer gets it, and
  the model has to read the row back to learn what the value was.
- **SQLite stores what it has types for.** It has no type for a timezone, so an aware `datetime` goes
  in and a naive one comes out, with no warning. Store UTC and say so, or use a database that has a
  type for it.
- **A value the model never checked is still written.** A table model's constructor does not
  validate, so a status outside the enumerated list reaches the table and fails on the way back.

### Where this shows up

Every model with a date in it, which is most of them. Money, where `Decimal` and the number of
digits are not optional. Anything with a status, a kind or a level. Ids made by the program rather
than the database, which is what UUIDs are for. The **SQLAlchemy, Deep Dive** guide's Column Types
notebook is the long version of this, with type decorators and what each database has; the
**sa_column and __table_args__** notebook is where a field with no matching column gets one.

### What this notebook covers

- What each Python type becomes as a column
- What comes back, and the timezone that does not
- Defaults: the model's and the table's
- `unique`, `index` and the lengths
- Ids the program makes
- A mission filed and read back, finished
- Four failures, from a field with no column to a row that cannot be read

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from datetime import date
from decimal import Decimal

from sqlmodel import Field, SQLModel


class Mission(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    title: str = Field(max_length=80)
    starts_on: date
    reward: Decimal = Field(default=Decimal("0.00"), max_digits=8, decimal_places=2)
    secret: bool = False


for column in Mission.__table__.columns:
    print(f"{column.name:<10} {str(column.type):<14} nullable {str(column.nullable):<5} "
          f"default {column.server_default}")
```

```
id         INTEGER        nullable False default None
title      VARCHAR(80)    nullable False default None
starts_on  DATE           nullable False default None
reward     NUMERIC(8, 2)  nullable False default None
secret     BOOLEAN        nullable False default None
```

Five annotations, five column types, and not one of them written twice. `Decimal` with `max_digits`
and `decimal_places` became `NUMERIC(8, 2)`, which is money stored as money. The last column is worth
as much as the rest: `secret` and `reward` have defaults in the model, and not one column in the
table has a default of its own, which is a distinction this notebook spends a section on.


## Setup

Seventeen imports, one of them installed first where it is missing, the cast, three helpers, the
classes, the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Session`, `create_engine` and `select`, from
  it, are the classes, the session and the engine. Colab does not have SQLModel, so the cell installs
  0.0.42 with `pip` where it is missing, and `version` and `PackageNotFoundError`, from
  `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `date`, `datetime` and `timezone`, from `datetime`, `Decimal`, from `decimal`, `Enum`, from `enum`,
  and `uuid` are the Python types this notebook gives to fields
- `event`, `insert`, `func` and `text`, from `sqlalchemy`, are the pragma on every connection, the
  rows loaded without a session, the `CURRENT_TIMESTAMP` behind a column default, and the two rows
  this notebook writes as plain SQL
- `ValidationError`, from `pydantic`, and `IntegrityError` and `StatementError`, from
  `sqlalchemy.exc`, are what the Common errors catch
- `CreateTable`, from `sqlalchemy.schema`, and `sqlite`, from `sqlalchemy.dialects`, write the
  `CREATE TABLE` a model describes
- `re` takes memory addresses out of a message, `Path` names the database file, and `shutil` removes
  the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads, and `mission_id` makes a UUID from a title
  with `uuid.uuid5`, so that every run of this notebook prints the same ids

`hero_engine` and `build` are the **Engine and create_all** and **Sessions** notebooks' engine and
loader. The heroes are here because a mission points at one.


In [1]:
import re
import shutil
import subprocess
import sys
import uuid
from datetime import date, datetime, timezone
from decimal import Decimal
from enum import Enum
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from pydantic import ValidationError
from sqlalchemy import event, func, insert, text
from sqlalchemy.dialects import sqlite
from sqlalchemy.exc import IntegrityError, StatementError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)


def mission_id(title):
    """The same id for the same title on every run: uuid4 would give a new one each time."""
    return uuid.uuid5(uuid.NAMESPACE_URL, f"https://example.edu/missions/{title}")


print("sqlmodel", sqlmodel.__version__, "| heroes:", len(HEROES), "| an id from a title:", mission_id("bridge"))


sqlmodel 0.0.42 | heroes: 8 | an id from a title: f0180a0a-90b4-5aa6-9a38-7ee1dd172be0


## Worked examples

### What each Python type becomes as a column

One class with eight fields, and every kind of thing a mission has to store:


In [2]:
class Status(str, Enum):
    planned = "planned"
    running = "running"
    complete = "complete"


class Mission(SQLModel, table=True):
    id: uuid.UUID = Field(default_factory=uuid.uuid4, primary_key=True)
    hero_id: int | None = Field(default=None, foreign_key="hero.id")
    title: str = Field(unique=True, max_length=80)
    status: Status = Field(default=Status.planned, index=True)
    starts_on: date = Field(default_factory=date.today)              # Python fills this in
    filed_at: datetime | None = Field(default=None, sa_column_kwargs={"server_default": func.now()})
    reward: Decimal = Field(default=Decimal("0.00"), max_digits=8, decimal_places=2)
    secret: bool = False


SQLModel.metadata.create_all(engine)
print(table_sql(Mission))


CREATE TABLE mission (
	id CHAR(32) NOT NULL, 
	hero_id INTEGER, 
	title VARCHAR(80) NOT NULL, 
	status VARCHAR(8) NOT NULL, 
	starts_on DATE NOT NULL, 
	filed_at DATETIME DEFAULT CURRENT_TIMESTAMP, 
	reward NUMERIC(8, 2) NOT NULL, 
	secret BOOLEAN NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(hero_id) REFERENCES hero (id), 
	UNIQUE (title)
)


Eight annotations and eight columns. `uuid.UUID` became a `CHAR(32)`, the hexadecimal digits of the
UUID with no dashes. The `Status` enumerated type became `VARCHAR(8)`, wide enough for `complete`,
its longest value. `Decimal` with `max_digits=8` and `decimal_places=2` became `NUMERIC(8, 2)`, which
holds a reward up to 999,999.99 exactly, where a `float` would have stored something close to it.
`unique=True` on the title is a constraint in the table, and `starts_on` and `filed_at` are the two
defaults, which have their own section below.

### What comes back, and the timezone that does not

A mission written and read in another session, with every value printed beside its type:


In [3]:
FOUND = datetime(2026, 3, 1, 9, 30, tzinfo=timezone.utc)            # a moment, with its timezone

with Session(engine) as session:
    session.add(Mission(id=mission_id("bridge"), hero_id=1, title="Rescue the bridge",
                        status=Status.running, starts_on=date(2026, 3, 1), filed_at=FOUND,
                        reward=Decimal("1250.50"), secret=True))
    session.commit()

with Session(engine) as session:
    mission = session.exec(select(Mission)).one()
    for name, value in fields(mission).items():
        print(f"  {name:<10} {value!r:<46} {type(value).__name__}")


  id         UUID('f0180a0a-90b4-5aa6-9a38-7ee1dd172be0')   UUID
  hero_id    1                                              int
  title      'Rescue the bridge'                            str
  status     <Status.running: 'running'>                    Status
  starts_on  datetime.date(2026, 3, 1)                      date
  filed_at   datetime.datetime(2026, 3, 1, 9, 30)           datetime
  reward     Decimal('1250.50')                             Decimal
  secret     True                                           bool


Six of the eight came back as what went in: a `UUID`, a `Status` member rather than a plain string, a
`date`, a `Decimal` with its cents, a `bool` and an `int`. The one that changed is `filed_at`. It
went in at 09:30 UTC and came back at 09:30 with no timezone at all:


In [4]:
with Session(engine) as session:
    mission = session.exec(select(Mission)).one()
    print("written :", FOUND, "|", FOUND.tzinfo)
    print("read    :", mission.filed_at, "|", mission.filed_at.tzinfo)
    print("equal   :", mission.filed_at == FOUND.replace(tzinfo=None))


def as_utc(moment):
    """A moment read from SQLite, given back the timezone this program stores everything in."""
    return moment.replace(tzinfo=timezone.utc)


print("with the zone put back:", as_utc(mission.filed_at), "| equal to what went in:",
      as_utc(mission.filed_at) == FOUND)


written : 2026-03-01 09:30:00+00:00 | UTC
read    : 2026-03-01 09:30:00 | None
equal   : True
with the zone put back: 2026-03-01 09:30:00+00:00 | equal to what went in: True


SQLite writes a `DATETIME` as text and has nowhere to put the zone, so it is dropped on the way in
and cannot be recovered on the way out. The value is not wrong, it is unlabeled, and the program is
what remembers the label. Two ways to live with that, and the first is the one to take: store UTC
everywhere and attach it as values come back, as `as_utc` does. The second is a database with a type
for it, PostgreSQL's `TIMESTAMP WITH TIME ZONE`, where the same model needs no helper.

### Defaults: the model's and the table's

`starts_on` and `filed_at` are both filled in when nothing is passed, and they are filled in by
different things. One is in the `CREATE TABLE` and the other is not:


In [5]:
for line in table_sql(Mission).splitlines():
    if "starts_on" in line or "filed_at" in line:
        print(line.strip())

with Session(engine) as session:
    quiet = Mission(id=mission_id("quiet"), title="Quiet week", hero_id=3)
    session.add(quiet)
    session.commit()
    session.refresh(quiet)
    print("starts_on is today  :", quiet.starts_on == date.today())
    print("filed_at was filled :", quiet.filed_at is not None, type(quiet.filed_at).__name__)
    print("status              :", quiet.status, "| reward:", quiet.reward, "| secret:", quiet.secret)


starts_on DATE NOT NULL,
filed_at DATETIME DEFAULT CURRENT_TIMESTAMP,
starts_on is today  : True
filed_at was filled : True datetime
status              : Status.planned | reward: 0.00 | secret: False


`starts_on` got today's date from `default_factory=date.today`, which ran in Python when the object
was built. `filed_at` got its value from the table: `server_default` put `DEFAULT CURRENT_TIMESTAMP`
in the column, the `INSERT` left the column out, and the database filled it. That is why the object
had to be refreshed to see it, and why a Python default needs no refresh.

| Where the default is | Written as | Who gets it | Read back |
|---|---|---|---|
| in the model | `Field(default=...)` or `default_factory=...` | rows built from the model | already in the object |
| in the table | `Field(sa_column_kwargs={"server_default": ...})` | every writer, models included | needs a refresh or a reread |

A Python default is the ordinary choice, and the one to reach for when the value is the program's
business. A column default is what to write when rows arrive from anywhere else: a migration
backfilling a new column, another service, an import, or somebody typing SQL. The second of the
Common errors is what happens when the first is used and rows arrive from elsewhere anyway.

### unique, index and the lengths

Three of the arguments in that class changed the schema in ways the columns do not show:


In [6]:
print("constraints:", sorted(type(constraint).__name__ + " on " + ", ".join(column.name for column in constraint.columns)
                             for constraint in Mission.__table__.constraints))
print("indexes    :", sorted(index.name for index in Mission.__table__.indexes))
print("lengths    :", [(column.name, column.type.length) for column in Mission.__table__.columns
                       if getattr(column.type, "length", None)])


constraints: ['ForeignKeyConstraint on hero_id', 'PrimaryKeyConstraint on id', 'UniqueConstraint on title']
indexes    : ['ix_mission_status']
lengths    : [('title', 80), ('status', 8)]


`unique=True` on the title is a `UniqueConstraint` in the table, so two missions cannot share a
title, and the database is what enforces it. `index=True` on the status is a separate index, named
`ix_mission_status`. The lengths are the two `VARCHAR` sizes: 80 from `max_length`, and 8 from
`complete`, the longest value of the enumerated type. The UUID column has no length of its own, since
`CHAR(32)` is the whole of its type. SQLite does not enforce a `VARCHAR` length, and PostgreSQL and
the others do, which is a good reason to set one that is honest.

### Ids the program makes

An `int` primary key is given out by the database, one row at a time. A UUID is made in Python
before the row exists, which is what lets a program know an id in advance, and lets two programs
write rows that will not collide:


In [7]:
made = Mission(title="Night watch", starts_on=date(2026, 4, 1))
before = made.id                                                    # uuid4 gave it one, so it is not printed here
print("id before any database saw it:", type(before).__name__, "| set:", before is not None)

with Session(engine) as session:
    session.add(made)
    session.commit()
    print("the same id after the commit :", made.id == before)
    print("an id made from a title      :", mission_id("bridge"))
    print("and the same title again     :", mission_id("bridge"))


id before any database saw it: UUID | set: True
the same id after the commit : True
an id made from a title      : f0180a0a-90b4-5aa6-9a38-7ee1dd172be0
and the same title again     : f0180a0a-90b4-5aa6-9a38-7ee1dd172be0


The id was there before the commit, and the commit did not change it, because
`default_factory=uuid.uuid4` had already run when the object was built. That is the difference from
an `int` key, which does not exist until the `INSERT`. The cost is that a UUID is
32 characters rather than an 8-byte number, and rows arrive in no particular order, which matters
once a table is large.

`uuid4` gives a new value every time, so a notebook that printed one would print something different
on every run. `mission_id` uses `uuid.uuid5`, which is a name turned into a UUID by a hash: the same
title gives the same id everywhere, forever, which is what the two last lines show.

### A mission filed and read back, finished

The pieces of this notebook in one function. `file_mission` takes what a form would send, as text,
validates it into a `Mission`, writes it, and hands back the row as the database now holds it, with
the timezone put back on the moment it was filed:


In [8]:
def file_mission(engine, raw):
    """Validate one mission, write it, and return what the database holds, with UTC put back."""
    mission = Mission.model_validate({**raw, "id": mission_id(raw["title"])})
    with Session(engine) as session:
        session.add(mission)
        session.commit()
        session.refresh(mission)                                    # the column defaults are only in the row
        filed = fields(mission)
    filed["filed_at"] = as_utc(filed["filed_at"]) if filed["filed_at"] else None
    return filed


filed = file_mission(engine, {"title": "Guard the vault", "hero_id": "5", "status": "complete",
                              "starts_on": "2026-05-04", "reward": "900.25", "secret": "false"})
for name, value in filed.items():
    shown = f"a moment the table chose, in {value.tzinfo}" if name == "filed_at" else repr(value)
    print(f"  {name:<10} {shown}")


  id         UUID('b7cf843d-5c4c-59da-8047-328b2ab5be98')
  hero_id    5
  title      'Guard the vault'
  status     <Status.complete: 'complete'>
  starts_on  datetime.date(2026, 5, 4)
  filed_at   a moment the table chose, in UTC
  reward     Decimal('900.25')
  secret     False


Every value arrived as text and none of them stayed text. `hero_id` is an `int`, `status` is a
`Status` member, `starts_on` is a `date`, `reward` is a `Decimal` with its cents, and `secret` is
`False`, which is what Pydantic makes of the string `"false"`. `filed_at` came from the table, so
its value is a different moment on every run and only its zone is printed here. The id is the one
`mission_id` made from the title, so this row has the same id on every machine that runs this
notebook.

### Where each part came from

| In `file_mission` | What it relies on | The section that showed it |
|---|---|---|
| `Mission.model_validate(...)` | text converted into the types the annotations name | the **table=True** notebook |
| `mission_id(raw["title"])` | an id the program makes, the same for one name | Ids the program makes |
| `session.refresh(mission)` | a column default that only the row knows | Defaults: the model's and the table's |
| `fields(mission)` | a model's values in the order the class declares them | the **table=True** notebook |
| `as_utc(...)` | a `datetime` that came back without its zone | What comes back, and the timezone that does not |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/04-field-types-and-defaults-solutions.ipynb).

**1.** Write a `Gadget` table model with an `id`, a `name` of at most 60 characters, a `bought_on`
date, a `cost` of at most 6 digits with 2 decimal places, and `in_service`, a boolean that is `True`
unless something says otherwise. Print its `CREATE TABLE`.


In [9]:
# your code here


**2.** Print every column of `Gadget` with its type, whether it accepts a null, and its length where
it has one.


In [10]:
# your code here


**3.** Write a gadget whose cost is `Decimal("249.99")`, read it back in another session, and print
the value and its type. Then print what `float("249.99") * 3` gives and what
`Decimal("249.99") * 3` gives.


In [11]:
# your code here


**4.** Add a `Condition` enumerated type with the values `new`, `worn` and `broken`. Redefine
`Gadget` with a `condition` field defaulting to `new` and a `checked_at` whose default is the
database's `CURRENT_TIMESTAMP`, after `SQLModel.metadata.clear()`, and build it in a database of its
own, `scratch/gadgets.db`. Print the two new columns.


In [12]:
# your code here


**5.** Write a gadget into `scratch/gadgets.db` with `INSERT` as plain SQL. Name every column the
table itself has no default for, leave `checked_at` out, and print what `condition` and `checked_at`
came back as.


In [13]:
# your code here


**6.** Write a function that takes a gadget and returns a dictionary a response could carry: the
name, the cost as a string with two decimal places, and `bought_on` as text in `YYYY-MM-DD`.


In [14]:
# your code here


## Common errors

### ValueError: <class 'dict'> has no matching SQLAlchemy type


In [15]:
class Debrief(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    notes: dict                                                     # no column type fits this


ValueError: <class 'dict'> has no matching SQLAlchemy type

Every field of a table model has to become a column, and a `dict` is not something a column holds.
The same happens with `list[str]`, with a set, and with any class of your own. It is raised while the
class is being defined, which is the best time for it.

There are three answers, and which one is right depends on the data. Store it as JSON in one column,
which is what `sa_column=Column(JSON)` does and what the **sa_column and __table_args__** notebook
covers. Give it a table of its own, one row per note, which the **Relationships** notebook covers.
Or, when it is genuinely one string, store it as one:


In [16]:
class Debrief(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    notes: str = Field(default="", max_length=2000)


SQLModel.metadata.create_all(engine)
print([f"{column.name} {column.type}" for column in Debrief.__table__.columns])


['id INTEGER', 'notes VARCHAR(2000)']


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: mission.starts_on


In [17]:
with engine.begin() as connection:                                  # a row written by something that is not the model
    connection.execute(text("INSERT INTO mission (id, title, status, reward, secret) "
                            "VALUES (:id, 'Sweep the docks', 'planned', 0, 0)"),
                       {"id": mission_id("docks").hex})


IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: mission.starts_on
[SQL: INSERT INTO mission (id, title, status, reward, secret) VALUES (?, 'Sweep the docks', 'planned', 0, 0)]
[parameters: ('95ef8c68a15b5026b7bc7e48d89a6225',)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

`starts_on` has a default, and the default is in Python. The `CREATE TABLE` says `DATE NOT NULL` and
nothing more, so a row written by anything other than a `Mission` object has no value for it and the
database refuses the row. A migration filling in a new column, an import script, another service and
a line of SQL typed by hand all look like this.

`filed_at`, whose default is the table's, is filled in for that same row without being named:


In [18]:
with engine.begin() as connection:
    connection.execute(text("INSERT INTO mission (id, title, status, starts_on, reward, secret) "
                            "VALUES (:id, 'Sweep the docks', 'planned', '2026-06-01', 0, 0)"),
                       {"id": mission_id("docks").hex})

with Session(engine) as session:
    docks = session.exec(select(Mission).where(Mission.title == "Sweep the docks")).one()
    print("starts_on:", docks.starts_on, "| filed_at came from the table:", docks.filed_at is not None)


starts_on: 2026-06-01 | filed_at came from the table: True


### sqlalchemy.exc.StatementError: (builtins.TypeError) SQLite Date type only accepts Python date objects as input.


In [19]:
with Session(engine) as session:
    session.add(Mission(id=mission_id("harbor"), title="Watch the harbor", starts_on="2026-07-01"))
    try:
        session.commit()
    except StatementError as error:                                 # its later lines list the values in no fixed order
        print(type(error).__name__ + ":", str(error).splitlines()[0])
    session.rollback()


StatementError: (builtins.TypeError) SQLite Date type only accepts Python date objects as input.


The date arrived as text, the constructor of a table model does not validate, and the column type is
what finally refuses it, at the commit rather than at the line with the mistake. `model_validate`
converts the same text into a `date` at the point the data arrives, which is where a service should
be doing it anyway:


In [20]:
with Session(engine) as session:
    harbor = Mission.model_validate({"id": mission_id("harbor"), "title": "Watch the harbor",
                                     "starts_on": "2026-07-01"})
    print("converted:", repr(harbor.starts_on))
    session.add(harbor)
    session.commit()


converted: datetime.date(2026, 7, 1)


### LookupError: 'urgent' is not among the defined enum values. Enum name: status. Possible values: planned, running, complete


In [21]:
with Session(engine) as session:
    session.add(Mission(id=mission_id("tower"), title="Hold the tower", status="urgent",
                        starts_on=date(2026, 8, 1)))
    session.commit()                                                # written without a word
    print("the row is in the table")

with Session(engine) as session:
    session.exec(select(Mission)).all()


the row is in the table


LookupError: 'urgent' is not among the defined enum values. Enum name: status. Possible values: planned, running, complete

The status is a string the enumerated type does not have. The constructor did not check it, the
column has no constraint in SQLite that would have refused it, and the row went in. The failure comes
on the way out, when SQLAlchemy turns the text back into a `Status` and finds nothing to turn it
into. Every read that touches the row fails, including reads of other rows that came back in the same
query, which is why one bad row can break a page that has nothing to do with it.

The fix at the edge is `model_validate`, which refuses the value where it arrived. The fix for the
row already written is to correct it:


In [22]:
try:
    Mission.model_validate({"title": "Hold the tower", "status": "urgent", "starts_on": "2026-08-01"})
except ValidationError as error:
    print(message(error).splitlines()[-1].strip())

with engine.begin() as connection:
    connection.execute(text("UPDATE mission SET status = 'planned' WHERE status = 'urgent'"))

with Session(engine) as session:
    print("missions readable again:", len(session.exec(select(Mission)).all()))


Input should be 'planned', 'running' or 'complete' [type=enum, input_value='urgent', input_type=str]
missions readable again: 7


Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [23]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- The annotation picks the column type: `date`, `datetime`, `bool`, a `str` enumerated type,
  `uuid.UUID` and `Decimal` all have one, and `| None` decides whether it accepts a null.
- `max_length`, `max_digits` and `decimal_places` set the sizes, `unique=True` is a constraint in the
  table, and `index=True` is an index of its own.
- A default written as `default` or `default_factory` is Python's and reaches rows built from the
  model; `server_default` is the table's and reaches every writer.
- SQLite has nowhere to keep a timezone, so an aware `datetime` comes back naive: store UTC and put
  it back on the way out.
- A value the constructor never checked still reaches the table, and an enumerated value outside the
  list is written happily and raises on every read afterwards.


## What is next

The **Reading Rows** notebook is about getting them out again: `session.exec`, `where` and the
conditions Python's `and` cannot express, `one` when two rows match and when none do, ordering and
paging, and the tuple that comes back when a query asks for more than one model.


---

&#8592; **Previous:** [Sessions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/03-sessions.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
